In [1]:
import torch

print(torch.backends.mps.is_available())
print(torch.device("mps"))

False
mps


In [2]:
# Use %pip so packages install into the *same* Python as this notebook kernel
# (shell `pip3` often points at a different interpreter than the kernel).
%pip install -q rank_bm25 transformers datasets scikit-learn tqdm torch numpy pandas scipy accelerate sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json

with open("data/train-claims.json") as f:
    train_data = json.load(f)

with open("data/dev-claims.json") as f:
    dev_data = json.load(f)

with open("data/evidence.json") as f:
    evidence_data = json.load(f)

In [4]:
print(len(train_data))
print(len(evidence_data))

1228
1208827


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from tqdm import tqdm

# evidence (shared by BM25 in the next cell)
eid_list = list(evidence_data.keys())
corpus = list(evidence_data.values())

# --- TF-IDF (disabled; use BM25 in the next cell) ---
# 构建 TF-IDF 稀疏矩阵
_tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=300000,
    ngram_range=(1, 2)
)

_tfidf_evidence_matrix = _tfidf_vectorizer.fit_transform(corpus)

# 检索函数：返回 TF-IDF 分数最高的 top-k evidence ids（默认 50，供 cross-encoder 再排序）
def _retrieve_tfidf_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    query_vec = _tfidf_vectorizer.transform([claim])
    scores = (query_vec @ _tfidf_evidence_matrix.T).toarray()[0]
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


In [6]:
import re
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def _tokenize_bm25(text):
    text = (text or "").lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    stop = ENGLISH_STOP_WORDS
    return [t for t in tokens if t not in stop]


# BM25 index (same role as the commented TF-IDF block above)
tokenized_corpus = [_tokenize_bm25(doc) for doc in tqdm(corpus, desc="Tokenize for BM25")]
tokenized_corpus = [t if t else ["_"] for t in tokenized_corpus]
bm25 = BM25Okapi(tokenized_corpus)


# BM25 检索（命名与诊断区 `_retrieve_tfidf_topk` 对齐）
def _retrieve_bm25_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    q_tokens = _tokenize_bm25(claim)
    if not q_tokens:
        q_tokens = ["_"]
    scores = bm25.get_scores(q_tokens)
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


retrieve_top_k_fast = _retrieve_bm25_topk


Tokenize for BM25: 100%|██████████| 1208827/1208827 [00:04<00:00, 276716.91it/s]


### 检索与超参诊断区（非正式训练）

从 **TF-IDF / BM25 vs gold** 到 **并集 vs gold** 为止，仅用于检索与召回的可视化调参；其下 **Cross-encoder 微调** 后，才是正式 **Cross-encoder 重排** 与 `build_dataset` / Trainer。须已执行 **BM25 gold** 与 **embedding 检索**。


### Gold evidences vs **TF-IDF** top-50（dev，`dev-claims.json`）

在 **`dev_data`** 上统计（不要用 `train_data`）。单独拟合 TF-IDF 仅用于看 gold 命中率；主检索路径为 BM25（`_retrieve_bm25_topk`，别名 `retrieve_top_k_fast`）。运行下方 cell 后得到 **`tf_idf_top50`**：`claim_id -> top-50 evidence ids`（仅含至少一条 gold 的 claim）。


In [7]:
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# TF-IDF built only for this diagnostic (same hyperparameters as the commented retrieval cell).
_tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=300000,
    ngram_range=(1, 2),
)
_tfidf_evidence_matrix = _tfidf_vectorizer.fit_transform(corpus)


def _retrieve_tfidf_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    query_vec = _tfidf_vectorizer.transform([claim])
    scores = (query_vec @ _tfidf_evidence_matrix.T).toarray()[0]
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


tf_idf_top50 = {}


def tfidf_topk_gold_stats(claims_data, k=50):
    """Gold evidence IDs found in TF-IDF top-k (per claim and pooled)."""
    tf_idf_top50.clear()
    total_gold = 0
    total_hits = 0
    claims_with_any_hit = 0
    hits_per_claim = []

    for cid, item in tqdm(claims_data.items(), desc=f"TF-IDF top-{k} vs gold"):
        claim = item["claim_text"]
        gold = item.get("evidences") or []
        gold_set = set(gold)
        if not gold_set:
            continue
        top_eids = _retrieve_tfidf_topk(claim, k=k)
        tf_idf_top50[cid] = top_eids
        top_set = set(top_eids)
        n_hit = len(gold_set & top_set)
        total_gold += len(gold_set)
        total_hits += n_hit
        hits_per_claim.append(n_hit)
        if n_hit > 0:
            claims_with_any_hit += 1

    n = len(hits_per_claim)
    if n == 0:
        print("No claims with non-empty gold evidences.")
        return
    mean_hits = sum(hits_per_claim) / n
    recall_at_k = total_hits / total_gold if total_gold else 0.0
    print(f"Claims with gold evidences: {n}")
    print(f"Total gold evidence links: {total_gold}")
    print(f"Gold links that appear in TF-IDF top-{k}: {total_hits}")
    print(f"Mean gold hits per claim in top-{k}: {mean_hits:.3f}")
    print(f"Recall@{k} (hits / all gold links): {recall_at_k:.3f}")
    print(f"Share of claims with at least one gold in top-{k}: {claims_with_any_hit / n:.3f}")


tfidf_topk_gold_stats(dev_data, k=50)


TF-IDF top-50 vs gold: 100%|██████████| 154/154 [00:26<00:00,  5.86it/s]

Claims with gold evidences: 154
Total gold evidence links: 491
Gold links that appear in TF-IDF top-50: 130
Mean gold hits per claim in top-50: 0.844
Recall@50 (hits / all gold links): 0.265
Share of claims with at least one gold in top-50: 0.539


### Gold evidences vs **BM25** top-50（dev，`dev-claims.json`）

在 **`dev_data`** 上统计（不要用 `train_data`）；使用上方 BM25 检索单元里的 **`_retrieve_bm25_topk`**（与 `retrieve_top_k_fast` 相同）。运行下方 cell 后得到 **`bm25_top50`**（`claim_id -> top-50 evidence ids`，仅含至少一条 gold 的 claim）。


In [8]:
from tqdm import tqdm


bm25_top50 = {}


def bm25_topk_gold_stats(claims_data, k=50):
    """Gold evidence IDs found in BM25 top-k (uses _retrieve_bm25_topk from the BM25 cell)."""
    bm25_top50.clear()
    total_gold = 0
    total_hits = 0
    claims_with_any_hit = 0
    hits_per_claim = []

    for cid, item in tqdm(claims_data.items(), desc=f"BM25 top-{k} vs gold"):
        claim = item["claim_text"]
        gold = item.get("evidences") or []
        gold_set = set(gold)
        if not gold_set:
            continue
        top_eids = _retrieve_bm25_topk(claim, k=k)
        bm25_top50[cid] = top_eids
        top_set = set(top_eids)
        n_hit = len(gold_set & top_set)
        total_gold += len(gold_set)
        total_hits += n_hit
        hits_per_claim.append(n_hit)
        if n_hit > 0:
            claims_with_any_hit += 1

    n = len(hits_per_claim)
    if n == 0:
        print("No claims with non-empty gold evidences.")
        return
    mean_hits = sum(hits_per_claim) / n
    recall_at_k = total_hits / total_gold if total_gold else 0.0
    print(f"Claims with gold evidences: {n}")
    print(f"Total gold evidence links: {total_gold}")
    print(f"Gold links that appear in BM25 top-{k}: {total_hits}")
    print(f"Mean gold hits per claim in top-{k}: {mean_hits:.3f}")
    print(f"Recall@{k} (hits / all gold links): {recall_at_k:.3f}")
    print(f"Share of claims with at least one gold in top-{k}: {claims_with_any_hit / n:.3f}")


bm25_topk_gold_stats(dev_data, k=50)


BM25 top-50 vs gold: 100%|██████████| 154/154 [05:01<00:00,  1.96s/it]

Claims with gold evidences: 154
Total gold evidence links: 491
Gold links that appear in BM25 top-50: 186
Mean gold hits per claim in top-50: 1.208
Recall@50 (hits / all gold links): 0.379
Share of claims with at least one gold in top-50: 0.682


### BM25 top-50 ∪ Embedding top-20 vs gold（dev，`dev-claims.json`）

本 cell **不再次检索**：仅读取 **`tf_idf_top50`** 与 **`bm25_top50`**，用 **`union_topk_vs_gold_sets`** / **`print_union_cached_vs_gold`** 统计并集与 gold。须先依次运行 TF-IDF gold 单元、BM25 gold 单元。


In [9]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


You can safely remove it manually.


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp314-cp314-win_amd64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp314-cp314-win_amd64.whl.metadata (5.6 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp314-cp314-win_amd64.whl.metadata (7.0 kB)
Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp314-cp314-win_amd64.whl (2771.0 MB)
Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp314-cp314-win_amd64.whl (9.7 MB)
Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp314-cp314-win_amd64.whl (1.7 MB)

   ---------------------------------------- 0/3 [torchaudio]
   ------------- -------------------------- 1/3 [torch]
   ------------- -----

In [10]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.11.0+cu128
12.8
True


In [13]:
from pathlib import Path

from tqdm import tqdm
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

if "bm25_top50" not in globals():
    raise RuntimeError("请先运行 BM25 gold 单元，生成 bm25_top50。")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    try:
        torch.cuda.set_device(0)
        print("CUDA device:", torch.cuda.get_device_name(torch.cuda.current_device()))
    except Exception:
        print("CUDA available but failed to query device name")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMB_ENCODE_BATCH = 256
EMBEDDING_TOPK = 20

_emb_cache_dir = Path("data/embedding_cache")
_emb_cache_dir.mkdir(parents=True, exist_ok=True)
_corpus_emb_path = _emb_cache_dir / "corpus_embeddings.npy"

print("Embedding retrieval device:", DEVICE)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

if _corpus_emb_path.is_file():
    _emb_np = np.load(_corpus_emb_path, mmap_mode="r")
    if _emb_np.shape[0] != len(eid_list):
        raise RuntimeError(
            f"缓存行数 {_emb_np.shape[0]} 与 evidence 条数 {len(eid_list)} 不一致，"
            f"请删 {_corpus_emb_path} 后重跑本单元。"
        )
    print("Loaded corpus embeddings:", _corpus_emb_path, _emb_np.shape)
    _evidence_embeddings = torch.from_numpy(np.asarray(_emb_np, dtype=np.float32)).to(DEVICE)
else:
    print(f"Building embeddings ({len(corpus)} docs) -> {_corpus_emb_path}")
    _emb_np = embedding_model.encode(
        corpus,
        batch_size=EMB_ENCODE_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(_corpus_emb_path, _emb_np)
    print("Saved corpus embeddings:", _corpus_emb_path, _emb_np.shape)
    _evidence_embeddings = torch.from_numpy(_emb_np).to(DEVICE)


def _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=None):
    if k <= 0:
        return []
    exclude_ids = set(exclude_ids or ())
    query_emb = embedding_model.encode(
        claim,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    if query_emb.dim() == 1:
        query_emb = query_emb.unsqueeze(0)
    query_emb = query_emb.to(DEVICE)
    # L2-normalized vectors: matmul == cosine similarity (all on GPU when available)
    scores = torch.matmul(query_emb, _evidence_embeddings.T)[0]
    n = len(eid_list)
    if n == 0:
        return []
    candidate_count = min(k + len(exclude_ids), n)
    _, top_idx = torch.topk(scores, k=candidate_count, largest=True, sorted=True)
    results = []
    for idx in top_idx.tolist():
        eid = eid_list[idx]
        if eid in exclude_ids:
            continue
        results.append(eid)
        if len(results) >= k:
            break
    return results


def union_topk_candidate_eids(bm_eids, emb_eids):
    """BM25 top-50 first, then embedding top-20 unique candidates."""
    return list(dict.fromkeys(list(bm_eids) + list(emb_eids)))


def union_topk_vs_gold_sets(bm_eids, emb_eids):
    B = set(bm_eids)
    E = set(emb_eids)
    return B, E, B | E


def print_union_cached_vs_gold(dev_data, bm25_top50):
    sg = su = sb = se = sm = sn = 0
    gold_ranks = []
    for cid, item in tqdm(dev_data.items(), desc="union-vs-gold"):
        G = set(item.get("evidences") or [])
        if not G:
            continue
        claim = item["claim_text"]
        bm_eids = bm25_top50.get(cid, ())
        emb_eids = _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=set(bm_eids))
        B, E, U = union_topk_vs_gold_sets(bm_eids, emb_eids)
        U_ordered = union_topk_candidate_eids(bm_eids, emb_eids)
        rank_in_u = {eid: i + 1 for i, eid in enumerate(U_ordered)}
        for eid in G:
            if eid in rank_in_u:
                gold_ranks.append(rank_in_u[eid])
        sg += len(G)
        su += len(G & U)
        sb += len(G & B & E)
        se += len(G & (E - B))
        sm += len(G & (B - E))
        sn += len(G - U)

    print("=== BM25 top-50 ∪ Embedding top-20 vs gold (dev_data, cached) ===")
    print("Total gold evidence links:", sg)
    print("Gold in union U:", su)
    print("  Both BM25 and Embedding:", sb)
    print("  Embedding only (in E not B):", se)
    print("  BM25 only (in B not E):", sm)
    print("Gold not in union:", sn)
    if sg:
        print(f"Union recall |G∩U|/|G|: {su / sg:.4f}")
        print(f"Both / |G|: {sb / sg:.4f}")
        print(f"Embedding only / |G|: {se / sg:.4f}")
        print(f"BM25 only / |G|: {sm / sg:.4f}")
        print(f"Outside union / |G|: {sn / sg:.4f}")
    print(
        "Partition check (both+embedding_only+bm25_only+outside == total gold links):",
        sb + se + sm + sn == sg,
    )
    if gold_ranks:
        mean_gold_rank_in_union = sum(gold_ranks) / len(gold_ranks)
        print(
            f"mean_gold_rank_in_union: {mean_gold_rank_in_union:.3f} "
            f"(over {len(gold_ranks)} gold links in U, 1-based rank in BM25→Embedding list)"
        )
    else:
        print("mean_gold_rank_in_union: n/a (no gold in union)")


print_union_cached_vs_gold(dev_data, bm25_top50)


CUDA device: NVIDIA GeForce RTX 5060
Embedding retrieval device: cuda:0


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19174.17it/s]


Building embeddings (1208827 docs) -> data\embedding_cache\corpus_embeddings.npy


Batches: 100%|██████████| 4722/4722 [03:53<00:00, 20.20it/s]


Saved corpus embeddings: data\embedding_cache\corpus_embeddings.npy (1208827, 384)


union-vs-gold: 100%|██████████| 154/154 [00:01<00:00, 107.98it/s]

=== BM25 top-50 ∪ Embedding top-20 vs gold (dev_data, cached) ===
Total gold evidence links: 491
Gold in union U: 266
  Both BM25 and Embedding: 0
  Embedding only (in E not B): 80
  BM25 only (in B not E): 186
Gold not in union: 225
Union recall |G∩U|/|G|: 0.5418
Both / |G|: 0.0000
Embedding only / |G|: 0.1629
BM25 only / |G|: 0.3788
Outside union / |G|: 0.4582
Partition check (both+embedding_only+bm25_only+outside == total gold links): True
mean_gold_rank_in_union: 26.835 (over 266 gold links in U, 1-based rank in BM25→Embedding list)


### 一次性检索缓存（CE 微调 + DistilBERT 共用）

**阶段 1（下方 cell，只需跑一次 ~40min）**：对 train/dev 每条 claim 做 BM25@50 + Embedding@20 → 并集 `U`，写入 `data/cache/`：
- `train_retrieval_cache.pkl` — 全部 train claim（供 DistilBERT）
- `train_union_cache.pkl` — 含 gold 且 `U` 有效的子集（供 CE 微调）
- `dev_retrieval_cache.pkl` — dev claim

**阶段 2（CE 微调并 `save` 到 `ce_finetuned/` 之后）**：用缓存里的 `U` + **已微调 CE** 重排到 top-5，生成 `train_texts` / `dev_texts`（见正式训练区的 DistilBERT cell），不再重复 BM25 全库打分。

**CE 微调**仍用 `train_union_cache` 构造正负例（未微调 CE 挖难负例）。


In [16]:
import pickle
from pathlib import Path

# auto：Jupyter/Cursor 里用 widget 单行进度条；终端里仍是普通 tqdm
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

if "_retrieve_embedding_topk" not in globals():
    raise RuntimeError("请先运行 embedding 检索单元，定义 _retrieve_embedding_topk。")
if "_retrieve_bm25_topk" not in globals():
    raise RuntimeError("请先运行 BM25 检索单元，定义 _retrieve_bm25_topk。")

RETRIEVE_K = 50
EMBEDDING_TOPK = 20
CACHE_DIR = Path("data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_RETRIEVAL_PATH = CACHE_DIR / "train_retrieval_cache.pkl"
DEV_RETRIEVAL_PATH = CACHE_DIR / "dev_retrieval_cache.pkl"
TRAIN_UNION_CACHE_PATH = CACHE_DIR / "train_union_cache.pkl"


def _pickle_load(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _pickle_save(path, obj):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def _retrieve_union_for_claim(claim):
    bm50 = _retrieve_bm25_topk(claim, k=RETRIEVE_K)
    emb20 = _retrieve_embedding_topk(
        claim, k=EMBEDDING_TOPK, exclude_ids=set(bm50)
    )
    union_eids = union_topk_candidate_eids(bm50, emb20)
    return bm50, emb20, union_eids


_cache_ok = (
    TRAIN_RETRIEVAL_PATH.is_file()
    and DEV_RETRIEVAL_PATH.is_file()
    and TRAIN_UNION_CACHE_PATH.is_file()
)

if _cache_ok:
    train_retrieval_cache = _pickle_load(TRAIN_RETRIEVAL_PATH)
    dev_retrieval_cache = _pickle_load(DEV_RETRIEVAL_PATH)
    train_union_cache = _pickle_load(TRAIN_UNION_CACHE_PATH)
    print("Loaded retrieval caches from", CACHE_DIR)
else:
    train_retrieval_cache = {}
    train_union_cache = {}

    for cid, item in tqdm(
        train_data.items(),
        desc="train: BM25+emb union",
        total=len(train_data),
        mininterval=0.5,
    ):
        claim = item["claim_text"]
        bm50, emb20, union_eids = _retrieve_union_for_claim(claim)
        train_retrieval_cache[cid] = {
            "claim_text": claim,
            "claim_label": item.get("claim_label"),
            "bm50": bm50,
            "emb20": emb20,
            "union_eids": union_eids,
        }
        gold = set(item.get("evidences") or [])
        if not gold or not union_eids:
            continue
        gold_in_U = [eid for eid in union_eids if eid in gold]
        if not gold_in_U:
            continue
        non_gold = [eid for eid in union_eids if eid not in gold]
        if not non_gold:
            continue
        train_union_cache[cid] = {
            "claim_text": claim,
            "gold_evidences": sorted(gold),
            "union_candidate_eids": union_eids,
            "gold_in_union": sorted(gold_in_U),
            "non_gold_eids": non_gold,
        }

    dev_retrieval_cache = {}
    for cid, item in tqdm(dev_data.items(), desc="dev: BM25+emb union"):
        claim = item["claim_text"]
        bm50, emb20, union_eids = _retrieve_union_for_claim(claim)
        dev_retrieval_cache[cid] = {
            "claim_text": claim,
            "claim_label": item.get("claim_label"),
            "bm50": bm50,
            "emb20": emb20,
            "union_eids": union_eids,
        }

    _pickle_save(TRAIN_RETRIEVAL_PATH, train_retrieval_cache)
    _pickle_save(DEV_RETRIEVAL_PATH, dev_retrieval_cache)
    _pickle_save(TRAIN_UNION_CACHE_PATH, train_union_cache)
    print("Saved retrieval caches to", CACHE_DIR)

print(
    f"train_retrieval={len(train_retrieval_cache)}  "
    f"train_union_CE={len(train_union_cache)}  "
    f"dev_retrieval={len(dev_retrieval_cache)}"
)


dev: BM25+emb union: 100%|██████████| 154/154 [05:21<00:00,  2.09s/it]

Saved retrieval caches to data\cache
train_retrieval=1228  train_union_CE=916  dev_retrieval=154


In [21]:
import random

import numpy as np
import torch
from sentence_transformers import CrossEncoder, InputExample
from tqdm import tqdm

NUM_HARD_NEG = 8
NUM_RANDOM_NEG = 2
CE_MINE_BATCH = 32
CROSS_ENCODER_BASE = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if "train_union_cache" not in globals():
    raise RuntimeError("请先运行上一 cell，生成 train_union_cache。")

_ce_mine_kw = {}
if torch.cuda.is_available():
    _ce_mine_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_mine_kw["device"] = "mps"
ce_miner = CrossEncoder(CROSS_ENCODER_BASE, **_ce_mine_kw)
print("Hard-negative miner device:", _ce_mine_kw.get("device", "cpu (default)"))

rng = random.Random(42)


def pick_hard_negatives(claim, non_gold_eids, k):
    if not non_gold_eids or k <= 0:
        return []
    k = min(k, len(non_gold_eids))
    pairs = [(claim, evidence_data[eid]) for eid in non_gold_eids]
    scores = ce_miner.predict(pairs, show_progress_bar=False, batch_size=CE_MINE_BATCH)
    order = np.argsort(scores)[::-1][:k]
    return [non_gold_eids[i] for i in order]


ce_train_examples = []
n_pos = n_neg = 0

for cid, rec in tqdm(train_union_cache.items(), desc="build CE train pairs"):
    claim = rec["claim_text"]
    for eid in rec["gold_in_union"]:
        ce_train_examples.append(
            InputExample(texts=[claim, evidence_data[eid]], label=1.0)
        )
        n_pos += 1
    non_gold = rec["non_gold_eids"]
    if not non_gold:
        continue
    hard = pick_hard_negatives(claim, non_gold, NUM_HARD_NEG)
    hard_set = set(hard)
    pool = [e for e in non_gold if e not in hard_set]
    rand_k = min(NUM_RANDOM_NEG, len(pool))
    easy = rng.sample(pool, rand_k) if rand_k else []
    for eid in hard + easy:
        ce_train_examples.append(
            InputExample(texts=[claim, evidence_data[eid]], label=0.0)
        )
        n_neg += 1

print(f"Examples: {len(ce_train_examples)}  (positive={n_pos}, negative={n_neg})")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 20031.93it/s]


Hard-negative miner device: cuda


build CE train pairs: 100%|██████████| 916/916 [00:25<00:00, 36.55it/s]

Examples: 11062  (positive=1902, negative=9160)


In [22]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder

if "ce_train_examples" not in globals() or not ce_train_examples:
    raise RuntimeError("请先运行构造数据集 cell（ce_train_examples 为空）。")

CROSS_ENCODER_BASE = globals().get(
    "CROSS_ENCODER_BASE", "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CE_EPOCHS = 1
CE_BATCH_SIZE = 16
CE_LR = 2e-5

ce_train_dataloader = DataLoader(
    ce_train_examples, shuffle=True, batch_size=CE_BATCH_SIZE
)

cross_encoder = CrossEncoder(
    CROSS_ENCODER_BASE,
    num_labels=1,
    **_ce_mine_kw,
)

# 训练前快照：用于确认参数是否真的更新（v4 的 fit() 在部分环境下 save 仍是基座权重）
_param_snap = {
    n: p.detach().cpu().clone()
    for n, p in cross_encoder.model.named_parameters()
}

warmup = min(100, max(1, len(ce_train_examples) // CE_BATCH_SIZE))
# 使用 old_fit：直接在 self.model 上反传，避免 v4 CrossEncoderTrainer 训完但权重未写入 save 的问题
cross_encoder.old_fit(
    train_dataloader=ce_train_dataloader,
    epochs=CE_EPOCHS,
    warmup_steps=warmup,
    optimizer_params={"lr": CE_LR},
    show_progress_bar=True,
)

_max_delta = max(
    (p.detach().cpu() - _param_snap[n]).abs().max().item()
    for n, p in cross_encoder.model.named_parameters()
)
print(f"Max |param_after - param_before|: {_max_delta:.6e}")
if _max_delta < 1e-7:
    raise RuntimeError("训练后参数几乎未变，请勿 save；检查 device / loss / dataloader。")

CE_FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
cross_encoder.save(str(CE_FINETUNE_DIR))
print("cwd:", Path.cwd())
print("Fine-tuned CrossEncoder saved to:", CE_FINETUNE_DIR)
print("Files:", sorted(p.name for p in CE_FINETUNE_DIR.iterdir())[:8], "...")


Epoch: 100%|██████████| 1/1 [00:17<00:00, 17.25s/it]


Max |param_after - param_before|: 3.072727e-03


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.68it/s]

cwd: c:\Users\Administrator\text-classification
Fine-tuned CrossEncoder saved to: C:\Users\Administrator\text-classification\ce_finetuned
Files: ['README.md', 'config.json', 'config_sentence_transformers.json', 'model.safetensors', 'modules.json', 'sentence_bert_config.json', 'tokenizer.json', 'tokenizer_config.json'] ...


In [23]:
# 仅当上一 cell 已 old_fit 且 max param delta > 0 时再运行；勿对未训练的 cross_encoder save（会覆盖成基座）
from pathlib import Path

CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CE_FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
cross_encoder.save(str(CE_FINETUNE_DIR))
print(CE_FINETUNE_DIR, list(CE_FINETUNE_DIR.iterdir())[:5])

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.54it/s]

C:\Users\Administrator\text-classification\ce_finetuned [WindowsPath('C:/Users/Administrator/text-classification/ce_finetuned/config.json'), WindowsPath('C:/Users/Administrator/text-classification/ce_finetuned/config_sentence_transformers.json'), WindowsPath('C:/Users/Administrator/text-classification/ce_finetuned/model.safetensors'), WindowsPath('C:/Users/Administrator/text-classification/ce_finetuned/modules.json'), WindowsPath('C:/Users/Administrator/text-classification/ce_finetuned/README.md')]


### Cross-encoder 微调前后 vs gold（dev，BM25 top-50 ∪ Embedding top-20 并集内重排）

在同一候选集 `U`（BM25 top-50 ∪ Embedding top-20）上重排，重点看 **CE top-5 找回了 union 里多少 gold**。须已运行 dev 的 BM25 gold 单元与 embedding 检索单元且存在 `ce_finetuned/`。


In [25]:
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

CE_COMPARE_RETRIEVE_K = 50
CE_COMPARE_TOPK = 5
CE_COMPARE_BATCH = 32
CROSS_ENCODER_BASE = globals().get(
    "CROSS_ENCODER_BASE", "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
EMBEDDING_TOPK = 20

if "bm25_top50" not in globals() or "_retrieve_embedding_topk" not in globals():
    raise RuntimeError(
        "请先运行 BM25 gold 单元与 embedding 检索单元，生成 bm25_top50 和 _retrieve_embedding_topk。"
    )
if not (CE_FINETUNE_DIR / "config.json").is_file():
    raise RuntimeError(f"未找到微调权重: {CE_FINETUNE_DIR}，请先 CE 微调并 save。")

_ce_dev_kw = {}
if torch.cuda.is_available():
    _ce_dev_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_dev_kw["device"] = "mps"

ce_base = CrossEncoder(CROSS_ENCODER_BASE, **_ce_dev_kw)
ce_finetuned = CrossEncoder(str(CE_FINETUNE_DIR), **_ce_dev_kw)
print("Compare on dev_data | device:", _ce_dev_kw.get("device", "cpu"))


def _union_for_dev(cid, claim):
    bm = bm25_top50.get(cid, ())
    emb = _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=set(bm))
    if bm or emb:
        return union_topk_candidate_eids(bm, emb)
    bm = _retrieve_bm25_topk(claim, k=CE_COMPARE_RETRIEVE_K)
    emb = _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=set(bm))
    return union_topk_candidate_eids(bm, emb)


def _ce_rank_eids(model, claim, cand_eids):
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = model.predict(pairs, show_progress_bar=False, batch_size=CE_COMPARE_BATCH)
    order = np.argsort(scores)[::-1]
    return [cand_eids[i] for i in order]


def _eval_ce_gold(model, name):
    total_gold_links = 0
    hits_topk = 0
    total_gold_in_u = 0
    hits_gold_in_u_topk = 0
    claims_with_gold = 0
    claims_any_hit_topk = 0
    claims_any_hit_gold_in_u_topk = 0
    mrr_list = []
    gold_ranks = []

    for cid, item in tqdm(dev_data.items(), desc=f"CE {name}"):
        gold = set(item.get("evidences") or [])
        if not gold:
            continue
        claim = item["claim_text"]
        U = _union_for_dev(cid, claim)
        if not U:
            continue

        ranked = _ce_rank_eids(model, claim, U)
        top_set = set(ranked[:CE_COMPARE_TOPK])
        gold_in_u = gold & set(U)

        total_gold_links += len(gold)
        hits_topk += len(gold & top_set)
        total_gold_in_u += len(gold_in_u)
        hits_gold_in_u_topk += len(gold_in_u & top_set)
        claims_with_gold += 1
        if gold & top_set:
            claims_any_hit_topk += 1
        if gold_in_u & top_set:
            claims_any_hit_gold_in_u_topk += 1

        best_rank = None
        for g in gold_in_u:
            r = ranked.index(g) + 1
            gold_ranks.append(r)
            if best_rank is None or r < best_rank:
                best_rank = r
        if best_rank is not None:
            mrr_list.append(1.0 / best_rank)

    recall_topk = hits_topk / total_gold_links if total_gold_links else 0.0
    recall_gold_in_u_topk = (
        hits_gold_in_u_topk / total_gold_in_u if total_gold_in_u else 0.0
    )
    claim_hit_rate = claims_any_hit_topk / claims_with_gold if claims_with_gold else 0.0
    claim_hit_gold_in_u = (
        claims_any_hit_gold_in_u_topk / claims_with_gold if claims_with_gold else 0.0
    )
    mrr = float(np.mean(mrr_list)) if mrr_list else 0.0
    mean_rank = float(np.mean(gold_ranks)) if gold_ranks else float("nan")
    return {
        "name": name,
        "claims": claims_with_gold,
        "gold_in_U": total_gold_in_u,
        f"gold_in_U_hit_top{CE_COMPARE_TOPK}": hits_gold_in_u_topk,
        f"recall@{CE_COMPARE_TOPK}_gold_in_U": recall_gold_in_u_topk,
        "total_gold_links": total_gold_links,
        f"gold_hit_top{CE_COMPARE_TOPK}": hits_topk,
        f"recall@{CE_COMPARE_TOPK}_all_gold": recall_topk,
        "claim_hit_rate": claim_hit_rate,
        "claim_hit_gold_in_U": claim_hit_gold_in_u,
        "mrr": mrr,
        "mean_gold_rank_in_U": mean_rank,
        "gold_in_U_ranks_n": len(gold_ranks),
    }


stats_base = _eval_ce_gold(ce_base, "base")
stats_ft = _eval_ce_gold(ce_finetuned, "finetuned")

print(f"\n=== CE top-{CE_COMPARE_TOPK} vs gold (dev), U = BM25@50 U Embedding@20 ===")
print()
print("Gold in U: how many hit CE top-k?  |G∩U∩topk| / |G∩U|")
for label, st in [("base", stats_base), ("finetuned", stats_ft)]:
    hit = st[f"gold_in_U_hit_top{CE_COMPARE_TOPK}"]
    tot = st["gold_in_U"]
    rec = st[f"recall@{CE_COMPARE_TOPK}_gold_in_U"]
    print(f"  {label:10s}  {hit}/{tot}  recall={rec:.4f}")
b = stats_base[f"recall@{CE_COMPARE_TOPK}_gold_in_U"]
f = stats_ft[f"recall@{CE_COMPARE_TOPK}_gold_in_U"]
print(f"  delta (finetuned-base) on gold-in-U: {f - b:+.4f}")
print()
print("All gold links vs CE top-k:  |G∩topk| / |G|")
for label, st in [("base", stats_base), ("finetuned", stats_ft)]:
    hit = st[f"gold_hit_top{CE_COMPARE_TOPK}"]
    tot = st["total_gold_links"]
    rec = st[f"recall@{CE_COMPARE_TOPK}_all_gold"]
    print(f"  {label:10s}  {hit}/{tot}  recall={rec:.4f}")
print()
for key in ("claim_hit_gold_in_U", "claim_hit_rate", "mrr", "mean_gold_rank_in_U"):
    b, f = stats_base[key], stats_ft[key]
    delta = (b - f) if key == "mean_gold_rank_in_U" else (f - b)
    sign = "+" if delta >= 0 else ""
    print(f"{key:28s}  base={b:.4f}  finetuned={f:.4f}  ({sign}{delta:.4f})")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 19926.79it/s]


Compare on dev_data | device: cuda


CE finetuned: 100%|██████████| 154/154 [00:05<00:00, 28.48it/s]


=== CE top-5 vs gold (dev), U = BM25@50 U Embedding@20 ===

Gold in U: how many hit CE top-k?  |G∩U∩topk| / |G∩U|
  base        127/266  recall=0.4774
  finetuned   148/266  recall=0.5564
  delta (finetuned-base) on gold-in-U: +0.0789

All gold links vs CE top-k:  |G∩topk| / |G|
  base        127/491  recall=0.2587
  finetuned   148/491  recall=0.3014

claim_hit_gold_in_U           base=0.5519  finetuned=0.6169  (+0.0649)
claim_hit_rate                base=0.5519  finetuned=0.6169  (+0.0649)
mrr                           base=0.4623  finetuned=0.5320  (+0.0697)
mean_gold_rank_in_U           base=11.4925  finetuned=9.5038  (+1.9887)


### CE top-5 证据分数明细（dev，154 claims）

对每条有 gold 的 dev claim：在 **U = BM25 top-50 ∪ Embedding top-20** 上用 **微调后 CE** 打分，输出 **top-5** 明细到 `ce_top5_scores_dev.json`。


In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

CE_SCORE_RETRIEVE_K = 50
CE_SCORE_TOPK = 5
CE_SCORE_BATCH = 32
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CROSS_ENCODER_BASE = globals().get(
    "CROSS_ENCODER_BASE", "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
EMBEDDING_TOPK = 20

if "bm25_top50" not in globals() or "_retrieve_embedding_topk" not in globals():
    raise RuntimeError("请先运行 dev BM25 gold 单元与 embedding 检索单元。")
if not (CE_FINETUNE_DIR / "config.json").is_file():
    raise RuntimeError(f"缺少 {CE_FINETUNE_DIR}，请先 CE 微调并 save。")

if "ce_finetuned" not in globals():
    _kw = {}
    if torch.cuda.is_available():
        _kw["device"] = "cuda"
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        _kw["device"] = "mps"
    ce_finetuned = CrossEncoder(str(CE_FINETUNE_DIR), **_kw)


def _union_for_claim(cid, claim):
    bm = bm25_top50.get(cid, ())
    emb = _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=set(bm))
    if bm or emb:
        return union_topk_candidate_eids(bm, emb)
    bm = _retrieve_bm25_topk(claim, k=CE_SCORE_RETRIEVE_K)
    emb = _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=set(bm))
    return union_topk_candidate_eids(bm, emb)


def _ce_topk_scored(model, claim, cand_eids, topk=CE_SCORE_TOPK):
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = model.predict(pairs, show_progress_bar=False, batch_size=CE_SCORE_BATCH)
    order = np.argsort(scores)[::-1][: min(topk, len(cand_eids))]
    return [(cand_eids[i], float(scores[i])) for i in order]


ce_top5_scores_dev = {}

for cid, item in tqdm(dev_data.items(), desc="CE top-5 scores"):
    gold = set(item.get("evidences") or [])
    if not gold:
        continue
    claim = item["claim_text"]
    U = _union_for_claim(cid, claim)
    if not U:
        continue
    top5 = _ce_topk_scored(ce_finetuned, claim, U)
    ce_top5_scores_dev[cid] = {
        "claim_text": claim,
        "gold_evidences": sorted(gold),
        "top5": [
            {"rank": r, "evidence_id": eid, "score": sc, "is_gold": eid in gold}
            for r, (eid, sc) in enumerate(top5, start=1)
        ],
    }

out_path = Path("ce_top5_scores_dev.json")
with open(out_path, "w") as f:
    json.dump(ce_top5_scores_dev, f, indent=2)

print(f"Claims with gold (scored): {len(ce_top5_scores_dev)}")
print(f"Saved -> {out_path.resolve()}")
print()

for cid in sorted(ce_top5_scores_dev.keys()):
    rec = ce_top5_scores_dev[cid]
    print(f"=== {cid} ===")
    print(rec["claim_text"][:120] + ("..." if len(rec["claim_text"]) > 120 else ""))
    for row in rec["top5"]:
        g = "GOLD" if row["is_gold"] else "    "
        print(f"  {row['rank']}. [{g}] {row['evidence_id']}  score={row['score']:.4f}")
    print()


### 正式训练（从 Cross-encoder 起）

**阶段 2**：用 `data/cache/` 里已算好的 BM25∪Embedding 并集，仅做 **CE 重排 → top-5** 生成 DistilBERT 的 `train_texts` / `dev_texts`（不再重复 BM25 全库打分）。

须已运行：**一次性检索缓存** cell、**CE 微调**（`ce_finetuned/`），以及下方 CrossEncoder 加载 cell。


In [26]:
import os

import numpy as np
import torch
from sentence_transformers import CrossEncoder

from pathlib import Path

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
EMBEDDING_TOPK = 20

_ce_kw = {}
if torch.cuda.is_available():
    _ce_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_kw["device"] = "mps"

_ce_has_finetuned = CE_FINETUNE_DIR.is_dir() and (CE_FINETUNE_DIR / "config.json").is_file()
_ce_load_path = str(CE_FINETUNE_DIR) if _ce_has_finetuned else CROSS_ENCODER_MODEL
print("cwd:", Path.cwd())
print("CrossEncoder load:", _ce_load_path, "(finetuned)" if _ce_has_finetuned else "(base)")
print("CrossEncoder device:", _ce_kw.get("device", "cpu (default)"))
cross_encoder = CrossEncoder(_ce_load_path, **_ce_kw)


def union_topk_candidate_eids(bm_eids, emb_eids):
    """BM25 top-k ∪ Embedding top-20 candidates：先去重，顺序为 BM25 在前，再补 Embedding 独有。"""
    return list(dict.fromkeys(list(bm_eids) + list(emb_eids)))


def retrieve_reranked(claim, retrieve_k=50, final_k=5):
    """BM25 top-k 与 Embedding top-20（去除 BM25 已有）并集，交 cross-encoder 重排到 final_k。"""
    if "_retrieve_embedding_topk" not in globals():
        raise RuntimeError(
            "Union rerank 需要 `_retrieve_embedding_topk`：请先运行 embedding 检索单元。"
        )
    bm_eids = _retrieve_bm25_topk(claim, k=retrieve_k)
    emb_eids = _retrieve_embedding_topk(claim, k=EMBEDDING_TOPK, exclude_ids=set(bm_eids))
    cand_eids = union_topk_candidate_eids(bm_eids, emb_eids)
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(scores)[::-1][:final_k]
    return [cand_eids[i] for i in order]


import inspect
print("[OK] BM25+Embedding rerank:", inspect.signature(retrieve_reranked))


cwd: c:\Users\Administrator\text-classification
CrossEncoder load: C:\Users\Administrator\text-classification\ce_finetuned (finetuned)
CrossEncoder device: cuda


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10689.37it/s]

[OK] BM25+Embedding rerank: (claim, retrieve_k=50, final_k=5)


In [27]:
from tqdm import tqdm


def build_dataset(data, retrieve_k=50, final_k=5):
    texts = []
    labels = []

    label_map = {
        "SUPPORTS": 0,
        "REFUTES": 1,
        "NOT_ENOUGH_INFO": 2,
        "DISPUTED": 3,
    }

    for cid, item in tqdm(data.items()):
        claim = item["claim_text"]
        label = item["claim_label"]

        eids = retrieve_reranked(claim, retrieve_k=retrieve_k, final_k=final_k)

        evidences = [evidence_data[eid] for eid in eids]

        input_text = claim + " [SEP] " + " ".join(evidences)

        texts.append(input_text)
        labels.append(label_map[label])

    return texts, labels


In [28]:
import pickle
from pathlib import Path

import numpy as np
from tqdm import tqdm

if "train_retrieval_cache" not in globals() or "dev_retrieval_cache" not in globals():
    raise RuntimeError("请先运行「一次性检索缓存」cell。")
if "cross_encoder" not in globals():
    raise RuntimeError("请先运行 CrossEncoder 加载 cell（需已微调 ce_finetuned/）。")

CACHE_DIR = Path("data/cache")
DISTILBERT_DATASET_PATH = CACHE_DIR / "distilbert_dataset.pkl"
CE_RERANK_BATCH = 32
FINAL_K = 5

LABEL_MAP = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3,
}


def _pickle_load(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _pickle_save(path, obj):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def _ce_rerank_from_retrieval_cache(retrieval_cache, final_k=5):
    reranked = {}
    for cid, rec in tqdm(retrieval_cache.items(), desc="CE rerank (cached union)"):
        claim = rec["claim_text"]
        cand = rec["union_eids"]
        if not cand:
            reranked[cid] = []
            continue
        pairs = [(claim, evidence_data[eid]) for eid in cand]
        scores = cross_encoder.predict(
            pairs, show_progress_bar=False, batch_size=CE_RERANK_BATCH
        )
        order = np.argsort(scores)[::-1][:final_k]
        reranked[cid] = [cand[i] for i in order]
    return reranked


def _texts_labels_from_reranked(data, reranked_eids):
    texts, labels = [], []
    for cid, item in data.items():
        eids = reranked_eids.get(cid, [])
        evidences = [evidence_data[eid] for eid in eids]
        texts.append(item["claim_text"] + " [SEP] " + " ".join(evidences))
        labels.append(LABEL_MAP[item["claim_label"]])
    return texts, labels


if DISTILBERT_DATASET_PATH.is_file():
    _bundle = _pickle_load(DISTILBERT_DATASET_PATH)
    train_texts = _bundle["train_texts"]
    train_labels = _bundle["train_labels"]
    dev_texts = _bundle["dev_texts"]
    dev_labels = _bundle["dev_labels"]
    print("Loaded DistilBERT dataset from", DISTILBERT_DATASET_PATH)
else:
    train_reranked_eids = _ce_rerank_from_retrieval_cache(
        train_retrieval_cache, final_k=FINAL_K
    )
    dev_reranked_eids = _ce_rerank_from_retrieval_cache(
        dev_retrieval_cache, final_k=FINAL_K
    )
    train_texts, train_labels = _texts_labels_from_reranked(
        train_data, train_reranked_eids
    )
    dev_texts, dev_labels = _texts_labels_from_reranked(dev_data, dev_reranked_eids)
    _pickle_save(
        DISTILBERT_DATASET_PATH,
        {
            "train_texts": train_texts,
            "train_labels": train_labels,
            "dev_texts": dev_texts,
            "dev_labels": dev_labels,
            "train_reranked_eids": train_reranked_eids,
            "dev_reranked_eids": dev_reranked_eids,
        },
    )
    print("Saved DistilBERT dataset to", DISTILBERT_DATASET_PATH)

print(f"train={len(train_texts)}  dev={len(dev_texts)}")

CE rerank (cached union): 100%|██████████| 154/154 [00:04<00:00, 38.25it/s]

Saved DistilBERT dataset to data\cache\distilbert_dataset.pkl
train=1228  dev=154


In [29]:
import transformers
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
print(transformers.__file__)

C:\Users\Administrator\AppData\Roaming\Python\Python314\site-packages\transformers\__init__.py


In [30]:
class ClaimDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

In [31]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

C:\Users\Administrator\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 22659.67it/s]
[transformers] D

In [32]:
train_dataset = ClaimDataset(train_texts, train_labels, tokenizer)
dev_dataset = ClaimDataset(dev_texts, dev_labels, tokenizer)

In [33]:
import gc
import itertools
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments

if "train_texts" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("请先运行 DistilBERT 数据 cell 与 tokenizer 单元。")
if "ClaimDataset" not in globals():
    raise RuntimeError("请先运行定义 ClaimDataset 的单元。")

# 网格：组合数 = len(LR) * len(EPOCHS) * len(MAX_LEN)（可自行增删）
LEARNING_RATES = [2e-5, 3e-5]
NUM_TRAIN_EPOCHS_LIST = [2, 3]
MAX_LENGTHS = [256, 512]
# LEARNING_RATES = [3e-5]
# NUM_TRAIN_EPOCHS_LIST = [3]
# MAX_LENGTHS = [512]

SWEEP_ROOT = Path("distilbert_hyperparam_sweep")
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
BACKBONE = "distilbert-base-uncased"


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float((preds == labels).mean()),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }


use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
print("Trainer accelerator:", "CUDA" if use_cuda else ("MPS" if use_mps else "CPU"))

sweep_results = []

for lr, epochs, max_len in itertools.product(
    LEARNING_RATES, NUM_TRAIN_EPOCHS_LIST, MAX_LENGTHS
):
    run_name = f"lr{lr}_ep{int(epochs)}_ml{int(max_len)}"
    run_dir = SWEEP_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"Run: {run_name}  ->  {run_dir}")
    print(f"  learning_rate={lr}  num_train_epochs={epochs}  max_length={max_len}")

    model = DistilBertForSequenceClassification.from_pretrained(
        BACKBONE, num_labels=4
    )
    train_ds = ClaimDataset(train_texts, train_labels, tokenizer, max_len=max_len)
    dev_ds = ClaimDataset(dev_texts, dev_labels, tokenizer, max_len=max_len)

    training_args = TrainingArguments(
        output_dir=str(run_dir / "trainer_output"),
        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        dataloader_pin_memory=use_cuda,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()

    save_dir = run_dir / "saved_model"
    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(save_dir))
    tokenizer.save_pretrained(str(save_dir))

    row = {
        "run_name": run_name,
        "save_dir": str(save_dir.resolve()),
        "learning_rate": lr,
        "num_train_epochs": epochs,
        "max_length": max_len,
    }
    row.update({k: float(v) for k, v in metrics.items()})
    sweep_results.append(row)

    with open(run_dir / "metrics.json", "w") as f:
        json.dump(row, f, indent=2)

    del trainer, model, train_ds, dev_ds
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
    elif use_mps:
        torch.mps.empty_cache()

with open(SWEEP_ROOT / "sweep_results.json", "w") as f:
    json.dump(sweep_results, f, indent=2)

best_hp = max(sweep_results, key=lambda r: r.get("eval_macro_f1", 0.0))
print("\nBest hyperparams (by dev macro_f1):", best_hp["run_name"])
print(
    f"  lr={best_hp['learning_rate']}  epochs={best_hp['num_train_epochs']}  "
    f"max_len={best_hp['max_length']}  macro_f1={best_hp.get('eval_macro_f1')}"
)
print("→ 下方「最终训练」cell 会从 base 权重重新训 freeze3 / full，不直接沿用 sweep 的 checkpoint。")


Trainer accelerator: CUDA

Run: lr2e-05_ep2_ml256  ->  distilbert_hyperparam_sweep\lr2e-05_ep2_ml256
  learning_rate=2e-05  num_train_epochs=2  max_length=256


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15456.03it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.229701,1.258266,0.441558,0.153153
2,1.185985,1.236509,0.461039,0.240062


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.185985,1.236509,2,0.461039,0.240062


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.78it/s]


...

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 23558.21it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.223089,1.249675,0.441558,0.153153
2,1.175897,1.200154,0.467532,0.230660
3,1.005132,1.185112,0.506494,0.364718


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.005132,1.185112,3,0.506494,0.364718


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.80it/s]



Best hyperparams (by dev macro_f1): lr3e-05_ep3_ml512
  lr=3e-05  epochs=3  max_len=512  macro_f1=0.36471765228990666
→ 下方「最终训练」cell 会从 base 权重重新训 freeze3 / full，不直接沿用 sweep 的 checkpoint。


### 最终训练（base 权重 + sweep 最优超参）

**思路**（DistilBERT-base 共 **6** 层 transformer，编号 0–5）：

| 方案 | 做法 | 适用 |
|------|------|------|
| **freeze3** | 冻结 embeddings + 底层 3 层，只训上层 3 层 + 分类头 | 数据较少、想稳一点 |
| **full** | 全部参数可训 | 数据够、想榨性能 |

- **Sweep** 只负责选 `lr / epochs / max_length`，不把 sweep 里的权重当最终模型（避免在 dev 上反复试出来的 checkpoint 过拟合）。
- 两种方案都从 **`distilbert-base-uncased`** 重新初始化，用同一套最优超参各训一遍，在 dev 上比 `macro_f1`，取更好者写入 `distilbert_final/best/`。

**渐进解冻（progressive）** 在下一 cell 单独执行：先 freeze3 训若干 epoch，再解冻全模型、**lr÷5** 继续训；最后与 freeze3 / full 一起打对比表并载入 dev 最优到 `model`。

In [34]:
import gc
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments

if "train_texts" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("请先运行 DistilBERT 数据 cell 与 tokenizer 单元。")
if "ClaimDataset" not in globals():
    raise RuntimeError("请先运行 ClaimDataset 定义 cell。")
if "best_hp" not in globals():
    sweep_path = Path("distilbert_hyperparam_sweep/sweep_results.json")
    if not sweep_path.is_file():
        raise RuntimeError("请先运行超参 sweep cell，或提供 sweep_results.json。")
    with open(sweep_path) as f:
        sweep_results = json.load(f)
    best_hp = max(sweep_results, key=lambda r: r.get("eval_macro_f1", 0.0))
    print("Loaded best_hp from", sweep_path)

BACKBONE = "distilbert-base-uncased"
FINAL_ROOT = Path("distilbert_final")
FINAL_ROOT.mkdir(parents=True, exist_ok=True)

BEST_LR = float(best_hp["learning_rate"])
BEST_EPOCHS = int(best_hp["num_train_epochs"])
BEST_MAX_LEN = int(best_hp["max_length"])
FREEZE_BOTTOM_LAYERS = 3  # embeddings + transformer.layer[0..2]

use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float((preds == labels).mean()),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }


def set_distilbert_trainable(model, freeze_bottom_n_layers=0):
    """0 = full fine-tune; 3 = freeze embeddings + bottom 3 of 6 DistilBERT layers."""
    for p in model.parameters():
        p.requires_grad = True
    if freeze_bottom_n_layers <= 0:
        return
    for p in model.distilbert.embeddings.parameters():
        p.requires_grad = False
    n_layers = len(model.distilbert.transformer.layer)
    for i in range(min(freeze_bottom_n_layers, n_layers)):
        for p in model.distilbert.transformer.layer[i].parameters():
            p.requires_grad = False


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def run_final_training(mode_name, freeze_bottom_n_layers):
    run_dir = FINAL_ROOT / mode_name
    run_dir.mkdir(parents=True, exist_ok=True)

    model = DistilBertForSequenceClassification.from_pretrained(BACKBONE, num_labels=4)
    set_distilbert_trainable(model, freeze_bottom_n_layers=freeze_bottom_n_layers)
    n_train, n_total = count_trainable_params(model)
    print(
        f"\n{'=' * 72}\n{mode_name}: freeze_bottom={freeze_bottom_n_layers}  "
        f"trainable={n_train:,} / {n_total:,} ({100 * n_train / n_total:.1f}%)\n"
        f"lr={BEST_LR}  epochs={BEST_EPOCHS}  max_len={BEST_MAX_LEN}"
    )

    tok = DistilBertTokenizer.from_pretrained(BACKBONE)
    train_ds = ClaimDataset(train_texts, train_labels, tok, max_len=BEST_MAX_LEN)
    dev_ds = ClaimDataset(dev_texts, dev_labels, tok, max_len=BEST_MAX_LEN)

    training_args = TrainingArguments(
        output_dir=str(run_dir / "trainer_output"),
        learning_rate=BEST_LR,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=BEST_EPOCHS,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        dataloader_pin_memory=use_cuda,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()

    save_dir = run_dir / "saved_model"
    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(save_dir))
    tok.save_pretrained(str(save_dir))

    row = {
        "mode": mode_name,
        "freeze_bottom_layers": freeze_bottom_n_layers,
        "save_dir": str(save_dir.resolve()),
        "learning_rate": BEST_LR,
        "num_train_epochs": BEST_EPOCHS,
        "max_length": BEST_MAX_LEN,
        "trainable_params": n_train,
        "total_params": n_total,
    }
    row.update({k: float(v) for k, v in metrics.items()})
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(row, f, indent=2)

    del trainer, model, train_ds, dev_ds
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
    elif use_mps:
        torch.mps.empty_cache()
    return row


final_results = []
final_results.append(run_final_training("freeze3", freeze_bottom_n_layers=FREEZE_BOTTOM_LAYERS))
final_results.append(run_final_training("full", freeze_bottom_n_layers=0))

with open(FINAL_ROOT / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("\nDone: freeze3 + full. See", FINAL_ROOT / "final_results.json")
print("→ 可选：运行下方「渐进解冻 + 方案对比」cell（含 progressive 与汇总表）。")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12933.41it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



freeze3: freeze_bottom=3  trainable=21,857,284 / 66,956,548 (32.6%)
lr=3e-05  epochs=3  max_len=512


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.221911,1.244463,0.441558,0.153153
2,1.197600,1.218452,0.448052,0.199736
3,1.100515,1.200329,0.474026,0.296889


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.100515,1.200329,3,0.474026,0.296889


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11649.87it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



full: freeze_bottom=0  trainable=66,956,548 / 66,956,548 (100.0%)
lr=3e-05  epochs=3  max_len=512


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.223089,1.249675,0.441558,0.153153
2,1.175920,1.200033,0.467532,0.230660
3,1.005190,1.184800,0.506494,0.364718


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.005190,1.184800,3,0.506494,0.364718


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.94it/s]



Done: freeze3 + full. See distilbert_final\final_results.json
→ 可选：运行下方「渐进解冻 + 方案对比」cell（含 progressive 与汇总表）。


### 渐进解冻（可选）+ 方案对比

**progressive**：阶段 1 冻底层 3 层、用 sweep 最优 `lr`；阶段 2 解冻全模型、`lr÷5` 继续训（总 epoch 略多于 sweep，属渐进配方）。

本 cell 可**单独运行**（需已有 `best_hp` 与 `train_texts`；若未跑上一 cell，会只对比磁盘上已有 `distilbert_final/*/metrics.json`）。

对比项：`freeze3` | `full` | `progressive`（+ 可选 sweep 基线），按 dev `macro_f1` 选最优并写入 `distilbert_final/best/`。

In [35]:
import gc
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments

RUN_PROGRESSIVE = True  # 设为 False 则只对比磁盘上已有 metrics，不重训 progressive

if "train_texts" not in globals():
    raise RuntimeError("请先运行 DistilBERT 数据 cell。")
if "ClaimDataset" not in globals():
    raise RuntimeError("请先运行 ClaimDataset 定义 cell。")

BACKBONE = "distilbert-base-uncased"
FINAL_ROOT = Path("distilbert_final")
SWEEP_ROOT = Path("distilbert_hyperparam_sweep")

if "best_hp" not in globals():
    with open(SWEEP_ROOT / "sweep_results.json") as f:
        best_hp = max(json.load(f), key=lambda r: r.get("eval_macro_f1", 0.0))
    print("Loaded best_hp from sweep_results.json")

BEST_LR = float(best_hp["learning_rate"])
BEST_EPOCHS = int(best_hp["num_train_epochs"])
BEST_MAX_LEN = int(best_hp["max_length"])
FREEZE_BOTTOM_LAYERS = 3
STAGE1_EPOCHS = max(1, int(round(BEST_EPOCHS * 0.33)))
STAGE2_EPOCHS = BEST_EPOCHS
STAGE2_LR = BEST_LR / 5.0

use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float((preds == labels).mean()),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }


def set_distilbert_trainable(model, freeze_bottom_n_layers=0):
    for p in model.parameters():
        p.requires_grad = True
    if freeze_bottom_n_layers <= 0:
        return
    for p in model.distilbert.embeddings.parameters():
        p.requires_grad = False
    for i in range(min(freeze_bottom_n_layers, len(model.distilbert.transformer.layer))):
        for p in model.distilbert.transformer.layer[i].parameters():
            p.requires_grad = False


def _train_model(model, tok, run_dir, lr, epochs, desc=""):
    train_ds = ClaimDataset(train_texts, train_labels, tok, max_len=BEST_MAX_LEN)
    dev_ds = ClaimDataset(dev_texts, dev_labels, tok, max_len=BEST_MAX_LEN)
    args = TrainingArguments(
        output_dir=str(run_dir / "trainer_output"),
        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        dataloader_pin_memory=use_cuda,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        compute_metrics=compute_metrics,
    )
    if desc:
        print(desc)
    trainer.train()
    metrics = trainer.evaluate()
    del trainer, train_ds, dev_ds
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
    elif use_mps:
        torch.mps.empty_cache()
    return metrics


def run_progressive_unfreeze():
    run_dir = FINAL_ROOT / "progressive"
    run_dir.mkdir(parents=True, exist_ok=True)
    tok = DistilBertTokenizer.from_pretrained(BACKBONE)

    print(
        f"\n{'=' * 72}\nprogressive: stage1 freeze{FREEZE_BOTTOM_LAYERS} "
        f"ep={STAGE1_EPOCHS} lr={BEST_LR}"
    )
    model = DistilBertForSequenceClassification.from_pretrained(BACKBONE, num_labels=4)
    set_distilbert_trainable(model, freeze_bottom_n_layers=FREEZE_BOTTOM_LAYERS)
    m1 = _train_model(
        model,
        tok,
        run_dir / "stage1",
        lr=BEST_LR,
        epochs=STAGE1_EPOCHS,
        desc="  [stage1] frozen bottom layers",
    )

    print(
        f"  [stage2] full unfreeze ep={STAGE2_EPOCHS} lr={STAGE2_LR} "
        f"(={BEST_LR}/5)"
    )
    set_distilbert_trainable(model, freeze_bottom_n_layers=0)
    m2 = _train_model(
        model,
        tok,
        run_dir / "stage2",
        lr=STAGE2_LR,
        epochs=STAGE2_EPOCHS,
    )

    save_dir = run_dir / "saved_model"
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(save_dir))
    tok.save_pretrained(str(save_dir))

    row = {
        "mode": "progressive",
        "freeze_bottom_layers": f"stage1={FREEZE_BOTTOM_LAYERS}, stage2=0",
        "save_dir": str(save_dir.resolve()),
        "stage1_epochs": STAGE1_EPOCHS,
        "stage2_epochs": STAGE2_EPOCHS,
        "stage1_lr": BEST_LR,
        "stage2_lr": STAGE2_LR,
        "max_length": BEST_MAX_LEN,
        "eval_macro_f1_stage1": float(m1.get("eval_macro_f1", 0)),
    }
    row.update({k: float(v) for k, v in m2.items()})
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(row, f, indent=2)
    del model
    gc.collect()
    return row


if RUN_PROGRESSIVE:
    run_progressive_unfreeze()
else:
    print("SKIP progressive training (RUN_PROGRESSIVE=False)")


def load_all_mode_metrics():
    rows = []
    for mode in ("freeze3", "full", "progressive"):
        p = FINAL_ROOT / mode / "metrics.json"
        if p.is_file():
            r = json.load(open(p))
            r.setdefault("mode", mode)
            rows.append(r)
    sweep_p = SWEEP_ROOT / "sweep_results.json"
    if sweep_p.is_file():
        sweep_best = max(json.load(open(sweep_p)), key=lambda x: x.get("eval_macro_f1", 0))
        rows.append(
            {
                "mode": "sweep_baseline",
                "save_dir": sweep_best.get("save_dir", ""),
                "eval_macro_f1": sweep_best.get("eval_macro_f1"),
                "eval_accuracy": sweep_best.get("eval_accuracy"),
                "learning_rate": sweep_best.get("learning_rate"),
                "num_train_epochs": sweep_best.get("num_train_epochs"),
                "max_length": sweep_best.get("max_length"),
                "note": "超参搜索时的一次 full 训练（非 final 配方）",
            }
        )
    return rows


all_rows = load_all_mode_metrics()
if not all_rows:
    raise RuntimeError("无 metrics：请先运行 freeze3/full cell，或设 RUN_PROGRESSIVE=True。")

compare_df = pd.DataFrame(
    [
        {
            "mode": r.get("mode"),
            "macro_f1": r.get("eval_macro_f1"),
            "accuracy": r.get("eval_accuracy"),
            "lr": r.get("learning_rate", r.get("stage2_lr")),
            "epochs": r.get("num_train_epochs", r.get("stage2_epochs")),
            "max_len": r.get("max_length"),
            "save_dir": r.get("save_dir", "")[:60],
        }
        for r in all_rows
    ]
).sort_values("macro_f1", ascending=False, na_position="last")

print("\n=== DistilBERT 方案对比（dev macro_f1 降序）===\n")
try:
    display(compare_df)
except NameError:
    print(compare_df.to_string(index=False))

best_row = max(
    [r for r in all_rows if r.get("mode") in ("freeze3", "full", "progressive")],
    key=lambda r: r.get("eval_macro_f1", 0.0),
)
best_save = Path(best_row["save_dir"])
best_link = FINAL_ROOT / "best"
if best_link.exists():
    shutil.rmtree(best_link) if best_link.is_dir() else best_link.unlink()
shutil.copytree(best_save, best_link)

model = DistilBertForSequenceClassification.from_pretrained(str(best_link))
tokenizer = DistilBertTokenizer.from_pretrained(str(best_link))
DISTILBERT_MAX_LEN_FOR_PREDICT = BEST_MAX_LEN

summary = {
    "best_mode": best_row["mode"],
    "best_macro_f1": best_row.get("eval_macro_f1"),
    "best_save_dir": str(best_link.resolve()),
    "all_modes": all_rows,
}
with open(FINAL_ROOT / "comparison_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nBest: {best_row['mode']}  macro_f1={best_row.get('eval_macro_f1'):.4f}")
print("Loaded `model` / `tokenizer` from:", best_link.resolve())


progressive: stage1 freeze3 ep=1 lr=3e-05


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14199.69it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [stage1] frozen bottom layers


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.222640,1.278786,0.454545,0.187007


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.222640,1.278786,1,0.454545,0.187007


...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.61it/s]



=== DistilBERT 方案对比（dev macro_f1 降序）===



,mode,macro_f1,accuracy,lr,epochs,max_len,save_dir
1,full,0.364718,0.506494,0.000030,3,512,C:\Users\Administrator\text-classification\dis...
3,sweep_baseline,0.364718,0.506494,0.000030,3,512,C:\Users\Administrator\text-classification\dis...
0,freeze3,0.296889,0.474026,0.000030,3,512,C:\Users\Administrator\text-classification\dis...
2,progressive,0.214450,0.441558,0.000006,3,512,C:\Users\Administrator\text-classification\dis...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 13811.91it/s]


Best: full  macro_f1=0.3647
Loaded `model` / `tokenizer` from: C:\Users\Administrator\text-classification\distilbert_final\best


In [36]:
import torch
import numpy as np
from tqdm import tqdm

use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
device = torch.device("cuda" if use_cuda else ("mps" if use_mps else "cpu"))
print("Predict device:", device)

model.to(device)
model.eval()

label_map_rev = {
    0: "SUPPORTS",
    1: "REFUTES",
    2: "NOT_ENOUGH_INFO",
    3: "DISPUTED",
}


def predict(data, retrieve_k=50, final_k=5):
    results = {}

    for cid, item in tqdm(data.items()):
        claim = item["claim_text"]

        eids = retrieve_reranked(claim, retrieve_k=retrieve_k, final_k=final_k)
        evidences = [evidence_data[eid] for eid in eids]

        input_text = claim + " [SEP] " + " ".join(evidences)

        max_len = int(globals().get("DISTILBERT_MAX_LEN_FOR_PREDICT", 512))
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_len,
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits.detach().float().cpu().numpy()
        pred = np.argmax(logits)

        results[cid] = {
            "claim_label": label_map_rev[pred],
            "evidences": eids,
        }

    return results


Predict device: cuda


In [ ]:
with open("data/test-claims-unlabelled.json") as f:
    test_data = json.load(f)
test_predictions = predict(test_data)
with open("test_predictions.json", "w") as f:
    json.dump(test_predictions, f, indent=2)

100%|██████████| 153/153 [03:23<00:00,  1.33s/it]


In [37]:
dev_predictions = predict(dev_data)
with open("dev_predictions.json", "w") as f:
    json.dump(dev_predictions, f, indent=2)

100%|██████████| 154/154 [05:15<00:00,  2.05s/it]


In [ ]:
# 分类器
print("DistilBERT:", getattr(model, "name_or_path", model.config._name_or_path))

# 重排 CE
print("CrossEncoder:", cross_encoder.model.config.name_or_path if hasattr(cross_encoder, "model") else cross_encoder)

# predict 用的 max_len
print("max_len:", globals().get("DISTILBERT_MAX_LEN_FOR_PREDICT", "not set"))

DistilBERT: /Users/zhaowenji/Text-Classification/distilbert_hyperparam_sweep/lr3e-05_ep3_ml512/saved_model
CrossEncoder: /Users/zhaowenji/Text-Classification/ce_finetuned
max_len: 512


In [38]:
class Args:

  def __init__(self):

    self.predictions = "dev_predictions.json"
    # self.predictions = "test_predictions.json"

    # self.predictions = "data/dev-claims-baseline.json"

    self.groundtruth = "data/dev-claims.json"

    self.verbose = True

In [39]:
import argparse
import sys
import json
import numpy as np

######
#main#
######

def main(args):

    try:
        predictions = json.load(open(args.predictions))
    except:
        print("Error loading predictions json file:", args.predictions)
        raise SystemExit

    try:
        groundtruth = json.load(open(args.groundtruth))
    except:
        print("Error loading groundtruth json file:", args.groundtruth)
        raise SystemExit

    try:
        f, acc = [], []

        #iterate through the groundtruth instances
        for claim_id, claim in sorted(groundtruth.items()):
            if claim_id in predictions and \
                "claim_label" in predictions[claim_id] and \
                "evidences" in predictions[claim_id]:

                #check claim level label
                instance_correct = 0.0
                if predictions[claim_id]["claim_label"] == claim["claim_label"]:
                    instance_correct = 1.0

                #check retrieved evidences
                evidence_correct = 0
                evidence_recall = 0.0
                evidence_precision = 0.0
                evidence_fscore = 0.0
                if type(predictions[claim_id]["evidences"]) == list and (len(predictions[claim_id]["evidences"]) > 0):
                    top_six_ev = set(predictions[claim_id]["evidences"])
                    for gr_ev in claim["evidences"]:
                        if gr_ev in top_six_ev:
                            evidence_correct += 1
                    if evidence_correct > 0:
                        evidence_recall = float(evidence_correct) / len(claim["evidences"])
                        evidence_precision = \
                            float(evidence_correct) / len(predictions[claim_id]["evidences"])
                        evidence_fscore = (2*evidence_precision*evidence_recall)/(evidence_precision+evidence_recall)

                if args.verbose:
                    print("groundtruth =", claim)
                    print("predictions =", predictions[claim_id])
                    print("instance accuracy =", instance_correct)
                    print("evidence recall =", evidence_recall)
                    print("evidence precision =", evidence_precision)
                    print("evidence fscore =", evidence_fscore, "\n\n")

                #add the metric results
                acc.append(instance_correct)
                f.append(evidence_fscore)

        #compute aggregate performance
        mean_f = np.mean(f if len(f) > 0 else [0.0])
        mean_acc = np.mean(acc if len(acc) > 0 else [0.0])
        if mean_f == 0.0 and mean_acc == 0.0:
            hmean = 0.0
        else:
            hmean = (2*mean_f*mean_acc)/(mean_f+mean_acc)

        print("Evidence Retrieval F-score (F)    =", mean_f)
        print("Claim Classification Accuracy (A) =", mean_acc)
        print("Harmonic Mean of F and A          =", hmean)

    except Exception as error:
        print("Error:", error)
        raise SystemExit

if __name__ == "__main__":

    #parser arguments
    desc = "Evaluation script that computes evidence retrieval f-score, claim classification accuracy, and aggregate performance."
    parser = argparse.ArgumentParser(description=desc)

    #arguments
    # parser.add_argument("--predictions", required=True, help="json file containing the claim label predictions and retrieved evidences produced by a system")
    # parser.add_argument("--groundtruth", required=True, help="json file containing the ground truth claim labels and evidences")
    # parser.add_argument("--verbose", action="store_true", help="turn on debug prints")
    # args = parser.parse_args()

    args = Args()

    main(args)

groundtruth = {'claim_text': 'The corals may save themselves, as many other creatures are attempting to do, by moving toward the poles as the Earth warms, establishing new reefs in cooler water.”', 'claim_label': 'SUPPORTS', 'evidences': ['evidence-242575', 'evidence-1175280']}
predictions = {'claim_label': 'SUPPORTS', 'evidences': ['evidence-1175280', 'evidence-1109959', 'evidence-940910', 'evidence-633104', 'evidence-900586']}
instance accuracy = 1.0
evidence recall = 0.5
evidence precision = 0.2
evidence fscore = 0.28571428571428575 


groundtruth = {'claim_text': 'Increases in atmospheric CO2 followed increases in temperature.', 'claim_label': 'SUPPORTS', 'evidences': ['evidence-368192', 'evidence-423643', 'evidence-629358']}
predictions = {'claim_label': 'SUPPORTS', 'evidences': ['evidence-714864', 'evidence-178206', 'evidence-368192', 'evidence-262028', 'evidence-121664']}
instance accuracy = 1.0
evidence recall = 0.3333333333333333
evidence precision = 0.2
evidence fscore = 0.25

# Retrieval:
1. TF-IDF top50
2. bm25 top50
3. union -> CE rerank top5
